#### Define date parameters

In [3]:
start_date = "2026-09-11"
end_date = "2026-09-17"

StatementMeta(, d742a4f8-7e92-46fc-9bab-062f76705111, 7, Finished, Available, Finished, False)

#### Import the required libraries

In [1]:
import reverse_geocoder as rg
from pyspark.sql.functions import (
col, lit, to_timestamp, when, udf
)
from pyspark.sql.types import StringType

StatementMeta(, d742a4f8-7e92-46fc-9bab-062f76705111, 5, Finished, Available, Finished, False)

#### Load recent Silver records

In [4]:
df = spark.table("earthquake_events_silver")
df = df.filter(
col("time") >= to_timestamp(lit(start_date))
)
display(df)

StatementMeta(, d742a4f8-7e92-46fc-9bab-062f76705111, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8ac0066d-4e4c-40de-aa1b-86b17935743e)

#### Add approximate country codes

In [8]:
def get_country_code(latitude, longitude):
    if latitude is None or longitude is None:
        return None
    coordinates = (latitude, longitude)
    result = rg.search(coordinates)
    return result[0].get("cc")

country_code_udf = udf(get_country_code, StringType())

df_gold = df.withColumn(
    "country_code",
    country_code_udf(col("latitude"), col("longitude"))
)

display(df_gold)

StatementMeta(, d742a4f8-7e92-46fc-9bab-062f76705111, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 425912c6-283b-468a-9207-8164070859ff)

#### Classify significance

In [9]:
df_gold = df_gold.withColumn(
    "sig_class",
    when(col("significance") < 100, "Low")
    .when(col("significance") < 500, "Moderate")
    .otherwise("High")
)

display(df_gold)

StatementMeta(, d742a4f8-7e92-46fc-9bab-062f76705111, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 892ab1e0-a977-4fce-9444-3ba9cd44ecfb)

#### Save the Gold Delta table

In [10]:
(
    df_gold.write
    .format("delta")
    .mode("append")
    .saveAsTable("earthquake_events_gold")
)

StatementMeta(, d742a4f8-7e92-46fc-9bab-062f76705111, 14, Finished, Available, Finished, False)